# COCODoom training on Colab

Run the cells top to bottom: config -> Drive -> repo -> deps -> dataset -> **train** -> KPI.
Resuming after a disconnect: mount Drive, re-run repo/deps/dataset, then set `RUN_DIR` in the config
cell to the `RESUMABLE` run dir printed by the scan cell and run the train cell again.
A fresh run costs one full pass of the cells again; checkpoints and logs live on Drive, so nothing is lost.

In [ ]:
# --- what to run ------------------------------------------------------------------
CONFIG = "config/exp_od_small_fcnn_colab.yml"  # GPU run; use config/exp_od_cpu_colab.yml for a CPU/smoke run
REPO_URL = "https://github.com/csxeba/verres.git"  # empty string = use REPO_ZIP instead
BRANCH = "feature/set_baselines"
REPO_ZIP = "/content/drive/MyDrive/verres.zip"  # fallback when the clone is not possible

# --- where the data lives --------------------------------------------------------
DATASET_URL = "https://www.robots.ox.ac.uk/~vgg/share/cocodoom-v1.0.1.tar.gz"  # official archives, ~2.2 GB
DATASET_ROOT = "/content/cocodoom"
REPO_DIR = "/content/verres"

# --- training ---------------------------------------------------------------------
RUN_DIR = None  # None = fresh run, otherwise a run dir printed as RESUMABLE by the scan cell
EPOCHS = None  # None keeps training.epochs from CONFIG, an int overrides it


## 1. Drive - the artifactory (checkpoints, logs) lives here; the dataset does not.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 2. Code - clone `REPO_URL`/`BRANCH`, or unzip `REPO_ZIP` when the clone fails.

In [ ]:
import os
import shutil
import subprocess

if not os.path.isdir(REPO_DIR):
    cloned = False
    if REPO_URL:
        result = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR],
                                capture_output=True, text=True)
        cloned = result.returncode == 0
        if not cloned:
            print("clone failed, falling back to REPO_ZIP:", result.stderr.strip()[-300:])
    if not cloned:
        # A Drive zip wraps the repo in one top-level folder, which has to disappear.
        tmp = "/content/_repo_unzip"
        shutil.rmtree(tmp, ignore_errors=True)
        shutil.unpack_archive(REPO_ZIP, tmp)
        entries = os.listdir(tmp)
        shutil.move(os.path.join(tmp, entries[0]) if len(entries) == 1 else tmp, REPO_DIR)

os.chdir(REPO_DIR)
print(os.getcwd())
print(sorted(os.listdir())[:12])
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip()
      if os.path.isdir(".git") else "no .git (zip checkout)")


## 3. Dependencies - Colab already ships a TF 2.15 runtime, only the extras are installed.

In [ ]:
!pip install -q -r requirements-colab.txt

import tensorflow as tf

print("tensorflow", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", [gpu.name for gpu in gpus])
if not gpus:
    print("WARNING: no GPU is visible to TensorFlow - training will be unusably slow."
          " Runtime -> Change runtime type -> T4 GPU.")


## 4. Dataset - download the official COCODoom archive into the VM (`DATASET_ROOT`), once per runtime.


In [ ]:
import json
import os
import subprocess

# The archive's inner top-level folder is `cocodoom/`, so strip-components lands the splits in DATASET_ROOT.
if not os.path.exists(os.path.join(DATASET_ROOT, "run-train.json")):
    archive = os.path.join("/content", os.path.basename(DATASET_URL))
    if not os.path.exists(archive):
        subprocess.run(["wget", "-c", "-q", "--show-progress", "-O", archive, DATASET_URL], check=True)
    os.makedirs(DATASET_ROOT, exist_ok=True)
    subprocess.run(["tar", "-xzf", archive, "-C", DATASET_ROOT, "--strip-components=1"], check=True)
    os.remove(archive)  # the frames are extracted, the 2.2 GB archive is not needed again

# The colab configs point `root` at DATASET_ROOT, so the splits have to sit at its top level.
for split in ("run-train", "run-val", "run-test"):
    path = os.path.join(DATASET_ROOT, f"{split}.json")
    assert os.path.isfile(path), f"missing {path}"

# The archive may ship a partial frame set; the annotation has to resolve against what was extracted.
images = json.load(open(os.path.join(DATASET_ROOT, "run-train.json")))["images"]
missing = [image["file_name"] for image in images[:50]
           if not os.path.isfile(os.path.join(DATASET_ROOT, image["file_name"]))]
assert not missing, f"frames referenced by run-train.json are missing, first: {missing[:3]}"

print(len(images), "run-train images, first:", images[0]["file_name"])
print(subprocess.run(["du", "-sh", DATASET_ROOT], capture_output=True, text=True).stdout.strip())


## 5. Resume scan - existing runs and whether they can be continued.

In [ ]:
import glob
import os

import yaml

context = yaml.safe_load(open(CONFIG))["context"]
runs_root = os.path.join(context["artifactory_root"], context["experiment_set"], context["experiment_name"])
print("runs root:", runs_root)

runs = sorted(glob.glob(os.path.join(runs_root, "xp_*")))
for run in runs:
    checkpoints = os.path.join(run, "checkpoints", "tf_ckpt")
    resumable = os.path.isdir(checkpoints) and bool(os.listdir(checkpoints))
    print(("RESUMABLE " if resumable else "          ") + run)
if not runs:
    print("no runs yet - the first training cell will create one")


**To resume**: set `RUN_DIR` in the config cell to one of the `RESUMABLE` paths above and run the
training cell. `main.main(continue_train=RUN_DIR)` re-reads that run's `config.yml`, restores the
newest checkpoint in `checkpoints/tf_ckpt` and continues from the epoch it recorded.
`EPOCHS` raises `training.epochs` when the resumed run needs more epochs than it was configured for.
Leave `RUN_DIR = None` to start a fresh run into a new `xp_<timestamp>` directory.

## 6. Train.

In [ ]:
import os
import sys

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

import main

updates = {"training.epochs": EPOCHS} if EPOCHS else None
if RUN_DIR:
    print("resuming", RUN_DIR)
    main.main(continue_train=RUN_DIR, config_updates=updates)
else:
    print("fresh run from", CONFIG)
    main.main(config_path=CONFIG, config_updates=updates)


## 7. KPIs - newest COCO eval KPI plus the tail of the training log.

In [ ]:
import glob
import json
import os

import pandas as pd

run = RUN_DIR or sorted(glob.glob(os.path.join(runs_root, "xp_*")))[-1]
print("run:", run)

kpi_files = sorted(glob.glob(os.path.join(run, "**", "*-kpi.json"), recursive=True), key=os.path.getmtime)
if kpi_files:
    print("kpi:", kpi_files[-1])
    print(json.dumps(json.load(open(kpi_files[-1])), indent=2))
else:
    print("no *-kpi.json in this run - it is written by an execution_type: evaluation run")

log_path = os.path.join(run, "training_logs.csv")
print(pd.read_csv(log_path).tail() if os.path.exists(log_path) else "no training_logs.csv yet")
